# Part 1 — Prepare the raw Airbnb review data

> Reorganized from `Airbnb_analysis_main.ipynb`. The original notebook is unchanged. Generated as one of four focused, independently usable parts.


## Centralized path

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Project folder
BASE_DIR = Path.cwd()


# Source and intermediate files
RAW_FILE = BASE_DIR / "data/raw/reviews_full.csv"
CLEAN_LINE_FILE = BASE_DIR / "data/processed/reviews_clean_line_terminators.csv"
REVIEWS_2023_2025_FILE = BASE_DIR / "data/processed/reviews_2023_2025.csv"
BASIC_CLEAN_FILE = BASE_DIR / "data/processed/reviews_2023_2025_basic_clean.csv"
STRATIFIED_FILE = BASE_DIR / "data/samples/reviews_sample_50000_stratified.csv"
EXTRACTION_READY_FILE = BASE_DIR / "data/samples/reviews_sample_50000_extraction_ready.csv"
TINY_TEST_FILE = BASE_DIR / "data/samples/reviews_tiny_test_24.csv"


print("Project folder:", BASE_DIR)

Project folder: c:\Users\yang9\Desktop\Capstone 406\Airbnb\python2


In [2]:
project_files = {
    "Raw data": RAW_FILE,
    "Line-terminator cleaned data": CLEAN_LINE_FILE,
    "2023–2025 data": REVIEWS_2023_2025_FILE,
    "Basic-clean data": BASIC_CLEAN_FILE,
    "Stratified sample": STRATIFIED_FILE,
    "Extraction-ready sample": EXTRACTION_READY_FILE,
    "24-comment test": TINY_TEST_FILE,
}

for name, path in project_files.items():
    print(f"{name}: {path.exists()}")

Raw data: True
Line-terminator cleaned data: True
2023–2025 data: True
Basic-clean data: True
Stratified sample: True
Extraction-ready sample: True
24-comment test: True


## Clean unusual line terminators

In [3]:
# Count separators before cleaning
def count_unusual_separators(file_path):
    """Count Unicode line and paragraph separators in a text file."""
    
    count_2028 = 0
    count_2029 = 0
    total_lines = 0

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as file:
        
        for line in file:
            total_lines += 1
            count_2028 += line.count("\u2028")
            count_2029 += line.count("\u2029")

    return {
        "total_lines": total_lines,
        "count_2028": count_2028,
        "count_2029": count_2029,
        "total_unusual": count_2028 + count_2029,
    }


original_counts = count_unusual_separators(RAW_FILE)

print("Original file inspection")
print("Total lines checked:", original_counts["total_lines"])
print("Number of \\u2028 found:", original_counts["count_2028"])
print("Number of \\u2029 found:", original_counts["count_2029"])
print("Total unusual separators found:", original_counts["total_unusual"])

Original file inspection
Total lines checked: 1841941
Number of \u2028 found: 118
Number of \u2029 found: 0
Total unusual separators found: 118


In [4]:
# Clean and count replacements
removed_2028 = 0
removed_2029 = 0

with open(
    RAW_FILE,
    "r",
    encoding="utf-8-sig",
    errors="replace",
    newline=""
) as input_file, open(
    CLEAN_LINE_FILE,
    "w",
    encoding="utf-8-sig",
    newline=""
) as output_file:

    for line in input_file:
        removed_2028 += line.count("\u2028")
        removed_2029 += line.count("\u2029")

        cleaned_line = line.replace("\u2028", " ")
        cleaned_line = cleaned_line.replace("\u2029", " ")

        output_file.write(cleaned_line)


total_removed = removed_2028 + removed_2029

print("Cleaning completed")
print("Removed \\u2028:", removed_2028)
print("Removed \\u2029:", removed_2029)
print("Total unusual separators removed:", total_removed)
print("Cleaned file saved:", CLEAN_LINE_FILE)

Cleaning completed
Removed \u2028: 118
Removed \u2029: 0
Total unusual separators removed: 118
Cleaned file saved: c:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data/processed/reviews_clean_line_terminators.csv


In [5]:
# Validate the cleaned file
cleaned_counts = count_unusual_separators(CLEAN_LINE_FILE)

print("Cleaned file validation")
print("Number of \\u2028 remaining:", cleaned_counts["count_2028"])
print("Number of \\u2029 remaining:", cleaned_counts["count_2029"])
print("Total unusual separators remaining:", cleaned_counts["total_unusual"])


assert cleaned_counts["count_2028"] == 0, \
    "\\u2028 characters still remain in the cleaned file."

assert cleaned_counts["count_2029"] == 0, \
    "\\u2029 characters still remain in the cleaned file."

assert total_removed == original_counts["total_unusual"], \
    "The number removed does not match the number originally detected."

print("Validation successful: all detected unusual separators were removed.")

Cleaned file validation
Number of \u2028 remaining: 0
Number of \u2029 remaining: 0
Total unusual separators remaining: 0
Validation successful: all detected unusual separators were removed.


In [6]:
# U+2029 is included as a defensive safeguard.

In [7]:
df_clean_terminators = pd.read_csv(
    CLEAN_LINE_FILE,
    keep_default_na=False,
    na_values=[""]
)
print("Rows:", len(df_clean_terminators))
print("Columns:", len(df_clean_terminators.columns))
print("Column names:")
print(df_clean_terminators.columns.tolist())

df_clean_terminators.head()

Rows: 1785848
Columns: 6
Column names:
['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments']


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,2708,13994902,2014-06-09,10905424,Kuberan,i had a wonderful stay. Everything from start ...
1,2708,14606598,2014-06-23,2247288,Camilla,Charles is just amazing and he made my stay sp...
2,2708,39597339,2015-07-25,27974696,Fallon,Staying with Chas was an absolute pleasure. He...
3,2708,61157407,2016-02-01,33226412,Haroon,Charles is a most wonderful host. I enjoyed my...
4,2708,66196280,2016-03-20,23408691,Massimo Litterio,Chas is a really good host. He gives me a lot ...


## Filter reviews from 2023 through 2025


In [8]:
df = df_clean_terminators.copy()

# Convert the date column to datetime for filtering
df["_date_parsed"] = pd.to_datetime(
    df["date"],
    errors="coerce"
)

# Check the original date data
invalid_date_count = df["_date_parsed"].isna().sum()

print("Invalid or missing dates:", invalid_date_count)
print("Earliest valid date in original data:", df["_date_parsed"].min())
print("Latest valid date in original data:", df["_date_parsed"].max())

# Safeguard 1: stop if any dates could not be converted
assert invalid_date_count == 0, \
    "Some date values could not be converted."

# Keep reviews from 2023-01-01 through 2025-12-31
df_2023_2025 = df[
    (df["_date_parsed"] >= "2023-01-01") &
    (df["_date_parsed"] < "2026-01-01")
].copy()

# Safeguard 2: stop if filtering produced no rows
assert not df_2023_2025.empty, \
    "No reviews were found between 2023 and 2025."

# Check the date range after filtering
print(
    "Earliest filtered date:",
    df_2023_2025["_date_parsed"].min()
)

print(
    "Latest filtered date:",
    df_2023_2025["_date_parsed"].max()
)

# Confirm that every retained row is within the required period
assert (
    df_2023_2025["_date_parsed"] >= "2023-01-01"
).all(), "Some reviews before 2023 remain."

assert (
    df_2023_2025["_date_parsed"] < "2026-01-01"
).all(), "Some reviews after 2025 remain."

# Remove the temporary helper column
df_2023_2025 = df_2023_2025.drop(
    columns=["_date_parsed"]
)

# Save the filtered file
df_2023_2025.to_csv(
    REVIEWS_2023_2025_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Original rows:", len(df))
print("Rows from 2023 to 2025:", len(df_2023_2025))
print("Filtered file saved:", REVIEWS_2023_2025_FILE)
print("Filtering validation successful.")

Invalid or missing dates: 0
Earliest valid date in original data: 2009-05-26 00:00:00
Latest valid date in original data: 2025-12-09 00:00:00
Earliest filtered date: 2023-01-01 00:00:00
Latest filtered date: 2025-12-09 00:00:00
Original rows: 1785848
Rows from 2023 to 2025: 898164
Filtered file saved: c:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data/processed/reviews_2023_2025.csv
Filtering validation successful.


In [9]:
#errors="coerce"
#If some date values are broken or impossible to understand, 
# do not crash. Convert them to NaT.
#NaT means Not a Time. It is like NaN, but for dates.

In [10]:
#Check the result
df_2023_2025.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
38,2708,871853918823721496,2023-04-17,155646940,Mamoun,A great host in a great home!<br/>Don’t hesita...
39,2708,947928431665277962,2023-07-31,502511212,Stephen,Charles is the man!! Just wrapped up an amazin...
40,2708,1107399330909692945,2024-03-07,201664381,Grant,Great starting point for anyone's LA adventure...
41,2708,1175511762719203533,2024-06-09,226451147,Hieu Trung,Very pleasant stay
42,2708,1215421255304446322,2024-08-03,68037230,Bailey,Had a great month long stay at Charles’s apart...


## Basic cleaning for stratifying and sampling


In [11]:
df_basic = df_2023_2025.copy()

# Create a temporary standardized version of the comments
# The original comments column remains unchanged
df_basic["_comment_check"] = (
    df_basic["comments"]
    .astype("string")
    .str.strip()
)

# Define clearly unusable comment categories

# 1. Truly missing values
missing_comments = df_basic["_comment_check"].isna()

# 2. Blank text after removing surrounding spaces
empty_comments = (
    df_basic["_comment_check"]
    .eq("")
    .fillna(False)
)

# 3. Placeholder text that contains no meaningful review
placeholder_values = {
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    ".",
    "-",
    "--",
}

placeholder_comments = (
    df_basic["_comment_check"]
    .str.lower()
    .isin(placeholder_values)
    .fillna(False)
)

# Combine all unusable comment conditions
bad_comments = (
    missing_comments |
    empty_comments |
    placeholder_comments
)

# Keep only rows containing usable comments
df_2023_2025_basic_clean = df_basic.loc[
    ~bad_comments
].copy()

# Remove the temporary helper column
df_2023_2025_basic_clean = (
    df_2023_2025_basic_clean
    .drop(columns=["_comment_check"])
)

# Calculate the number of removed rows
rows_removed = int(bad_comments.sum())

# Validate the row-count change
assert (
    len(df_2023_2025_basic_clean)
    == len(df_2023_2025) - rows_removed
), "The cleaned row count does not match the expected result."

# Confirm that the helper column was removed
assert "_comment_check" not in df_2023_2025_basic_clean.columns, \
    "The temporary helper column was not removed."

# Report the cleaning results
print("Rows before basic cleaning:", len(df_2023_2025))
print("Missing comments:", int(missing_comments.sum()))
print("Blank comments:", int(empty_comments.sum()))
print(
    "Placeholder/unusable comments:",
    int(placeholder_comments.sum())
)
print("Total rows removed:", rows_removed)
print(
    "Rows after basic cleaning:",
    len(df_2023_2025_basic_clean)
)
print("Basic cleaning validation successful.")


Rows before basic cleaning: 898164
Missing comments: 0
Blank comments: 0
Placeholder/unusable comments: 1448
Total rows removed: 1448
Rows after basic cleaning: 896716
Basic cleaning validation successful.


In [12]:
#The <NA> row is missing. 
#Black means ""   "     "
#palceholder: "None"   "N/A"   "-"   "."  "NULL"

In [13]:
# Save the basic-cleaned dataset
df_2023_2025_basic_clean.to_csv(
    BASIC_CLEAN_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Basic-cleaned file saved:", BASIC_CLEAN_FILE)

Basic-cleaned file saved: c:\Users\yang9\Desktop\Capstone 406\Airbnb\python2\data/processed/reviews_2023_2025_basic_clean.csv
